# Lilly v2 — listener pass-2 half 1 (whisper-large-v3)

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian speech only.

Pass-1 (Kaggle v12) was whisper-small on FLEURS hr only: 6,521 rows, 47% Bosnian,
38.4% → 33.9% WER. This pass changes the base **and** widens the Croatian side:
FLEURS hr plus voxpopuli_hr. Bosnian share stays **0.47**.

Two epochs plus BEFORE/AFTER WER miss Kaggle's 12h Save & Run All wall
(cancelled twice; the zip 404'd). This notebook is **half 1 of 2**: clone to
`/kaggle/temp`, train **one epoch**, zip the **Trainer adapter checkpoint**
(`lilly-listen-half1.zip`), COMPLETE. No BEFORE WER, no merge, no convert, no
AFTER WER.

Half 2 (`Lilly_Speech_Kaggle_Half2.ipynb`) attaches this Output and runs
`train_speech.py --resume` with `SPEECH_EPOCHS = 2` so Trainer continues epoch 2.
`--base` on merged weights would start a new LoRA — that is not epoch 2.

Set these in the panel on the right:

- **Session options → Accelerator → GPU T4**
- **Session options → Internet → On**

Then **Save Version → Save & Run All (Commit)** and close the tab.

Offload contract (`training/kaggle_offload.py`): Output always holds
`stdout.txt`, `experiment_log.json`, `metrics.jsonl`. COMPLETE + zip is still
not install. Not a Kaggle competition — do not submit.


In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co", "https://datasets-server.huggingface.co"):
    reachable(host)
print("network ok")

# Under /kaggle/working, so it is Output and outlives the log.
# agentic-kaggle-skill offload: the tee is stdout.txt, not a child fd Kaggle never sees.
TEE = Path("/kaggle/working/stdout.txt")

def run(*cmd, quiet=False):
    """Run a child process and put its output somewhere it can be found.

    Kaggle's log holds what this notebook process prints. A child process
    writing to its own stdout is not in it. OCR pass-7c logged the train
    command and then nothing until the zip. Speech had the same hole:
    subprocess.run(check=True) never reprints the child's loss or the
    encoder 0-grad line. Read the child's output here and reprint it, and
    tee it into Output so the numbers survive a dropped log too.
    """
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working
# Everything under /kaggle/working becomes Output. A clone there floods
# `kaggle kernels output` with git objects so the zip never downloads (OCR
# already learned this). Only the zip(s) below belong in Output.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "train_speech.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zips")
from training.kaggle_offload import Offload
OFF = Offload("speech", os.environ.get("LILLY_RUN_ID", "speech"))
OFF.hardware(torch.cuda.get_device_name(0))


In [ ]:
# 3. Install what we need (~3 min)
# The versions are read out of the repo's own requirements.txt rather than
# copied into this cell. A second, hand-kept list is exactly how the last run
# died: peft is pinned in requirements.txt and was simply absent from here, so
# Kaggle's own much newer peft got used instead — and that one's torchao
# dispatcher raises against the torchao Kaggle also ships. The first
# get_peft_model() call blew up, after a 3 GB download and forty minutes.
NEEDED = ["transformers", "accelerate", "peft", "faster-whisper", "ctranslate2",
          "soundfile", "scipy", "pyarrow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
unpinned = [n for n in NEEDED if n not in pins]
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin in requirements.txt, taking latest:", unpinned or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])


In [ ]:
# 3b. Prove this machine can actually train, before an hour is spent finding out
# Two things have to hold and neither shows up in a version number: peft must be
# able to build a LoRA layer on this image, and the GPU Kaggle handed us must
# actually run kernels. The last run satisfied "GPU is available" and still could
# not compute — Kaggle gave a P100 (sm_60) that the installed PyTorch does not
# support, and separately peft could not build a layer at all.
#
# So: build a real LoRA layer, put it on the GPU, push a gradient through it. It
# takes about twenty seconds and it fails here, loudly, instead of after the
# download.
import torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__)

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(32, 32)
    def forward(self, x):
        return self.q_proj(x)

tiny = get_peft_model(Tiny(), LoraConfig(r=4, target_modules=["q_proj"]))
try:
    tiny = tiny.cuda()
    out = tiny(torch.randn(4, 32, device="cuda")).sum()
    out.backward()
except RuntimeError as exc:
    raise SystemExit(
        f"The GPU cannot run this build of PyTorch ({exc}).\n"
        f"Card: {torch.cuda.get_device_name(0)}. Right panel -> Session options ->"
        f" Accelerator -> GPU T4 x2, then Save & Run All again.") from exc

lora = [n for n, p in tiny.named_parameters() if "lora_" in n and p.grad is not None]
assert lora, "peft built a LoRA layer but no gradient reached it"
print(f"LoRA trains on {torch.cuda.get_device_name(0)}: "
      f"{len(lora)} adapter tensors took a gradient")
del tiny
torch.cuda.empty_cache()


In [ ]:
# 4. Download the speech: clips to train on, and clips held back to judge with (~3 GB)
# The audio goes to scratch space, not into /kaggle/working — everything in the working
# directory is copied into the version Output, and 3 GB of wav files there is waste.
scratch = Path("/kaggle/temp/speech" if Path("/kaggle/temp").is_dir() else "/tmp/lilly-speech")
scratch.mkdir(parents=True, exist_ok=True)
if Path("data/speech").is_symlink():
    Path("data/speech").unlink()          # re-running the cell should not fail
if not Path("data/speech").exists():
    Path("data/speech").symlink_to(scratch)

run("python3", "data/scripts/download_speech_data.py")

for split in ("train", "valid", "test"):
    n = sum(1 for _ in open(f"data/speech/{split}.tsv", encoding="utf-8"))
    assert n > 100, f"{split}.tsv has only {n} clips — a download failed"
    print(f"{split}: {n:,} clips")


In [ ]:
# 4b. Croatian speech — downloaded here rather than uploaded from home.
# Kaggle's connection runs at roughly twenty times the one this project is
# developed on, and the audio is several gigabytes: pulling it here costs
# minutes where uploading it as a dataset would cost most of an evening.
# Scratch space for the same reason the Bosnian clips use it — anything left in
# /kaggle/working is copied into the Output, and gigabytes of wav there is waste.
extra = Path("/kaggle/temp/speech-extra" if Path("/kaggle/temp").is_dir()
             else "/tmp/lilly-speech-extra")
extra.mkdir(parents=True, exist_ok=True)
if Path("data/speech-extra").is_symlink():
    Path("data/speech-extra").unlink()
if not Path("data/speech-extra").exists():
    Path("data/speech-extra").symlink_to(extra)

# Two sources, two calls: a single --hours would override both defaults.
# fleurs_hr default is already 12 h (the whole train split). voxpopuli_hr
# default is 8 h of spontaneous EP speech FLEURS does not have.
run("python3", "data/scripts/download_extra_speech.py",
    "--source", "fleurs_hr", "--hours", "12")
run("python3", "data/scripts/download_extra_speech.py",
    "--source", "voxpopuli_hr")

fleurs_n = sum(1 for _ in open("data/speech-extra/fleurs_hr/train.tsv", encoding="utf-8"))
vox_path = Path("data/speech-extra/voxpopuli_hr/train.tsv")
assert fleurs_n > 500, f"only {fleurs_n} FLEURS hr clips — the download did not work"
assert vox_path.is_file(), (
    "voxpopuli_hr missing — this pass is the wider mix, not another FLEURS-only run")
vox_n = sum(1 for _ in open(vox_path, encoding="utf-8"))
assert vox_n > 200, f"only {vox_n} voxpopuli clips — the second source did not land"
print(f"Croatian FLEURS hr: {fleurs_n:,}  voxpopuli_hr: {vox_n:,}")


In [ ]:
# 4c. Mix them, keeping Bosnian at a stated share of the examples.
# Concatenating is the obvious move and the wrong one: with far more Croatian
# than Bosnian the model hears mostly Croatian and drifts towards it, and the
# overall error rate can fall while Bosnian gets worse.
#
# Pass-1 asked for 0.35 and landed at 47% because FLEURS hr was already that
# large a slice. This pass has more Croatian, so 0.35 would actually drown
# Bosnian. 0.47 is pass-1's measured mix and the floor the mixer holds by
# repeating Bosnian clips.
BOSNIAN_SHARE = 0.47
# Half 1 of 2: one epoch, then zip the Trainer checkpoint (adapter + optimizer).
# Merged large-v3 is ~3 GB and AFTER WER is what hit the 12h wall twice.
# Half 2 resumes this checkpoint with SPEECH_EPOCHS = 2 (Trainer continues
# epoch 2; it is not a new LoRA on merged weights).
SPEECH_EPOCHS = 1
SPEECH_BASE = "openai/whisper-large-v3"

run("python3", "data/scripts/build_speech_mix.py", "--share", str(BOSNIAN_SHARE))

MIX = "data/speech-extra/train-mix.tsv"
bosnian_only = sum(1 for _ in open("data/speech/train.tsv", encoding="utf-8"))
mixed = sum(1 for _ in open(MIX, encoding="utf-8"))
assert mixed > bosnian_only, (
    f"the mix has {mixed:,} rows against {bosnian_only:,} Bosnian — no Croatian "
    f"got in, so this run would not test what it is here to test")
budget = mixed * SPEECH_EPOCHS
assert budget <= 26_000, (
    f"{mixed:,} rows × {SPEECH_EPOCHS} epochs = {budget:,} row-epochs; "
    f"v14's clock says that misses the 12h wall. Cut voxpopuli hours, "
    f"do not silently drop epochs.")
print(f"mixed: {mixed:,} rows, from {bosnian_only:,} Bosnian, "
      f"{SPEECH_EPOCHS} epochs, {budget:,} row-epochs (budget 26,000)")


In [ ]:
# 5. Train one epoch and zip the Trainer checkpoint — then stop.
# No BEFORE WER (same untrained large-v3 every time, costs an hour).
# No merge (3 GB). No convert. No AFTER WER (that is what cancelled v16).
# --keep-adapter copies checkpoints-speech/checkpoint-* to listen-adapter
# with trainer_state.json so half 2 can resume_from_checkpoint.
os.environ["PYTHONUNBUFFERED"] = "1"
OFF.body["run_id"] = "speech-half1"
OFF.flush()
run("python3", "-u", "training/train_speech.py", "--data", MIX, "--base", SPEECH_BASE,
    "--epochs", str(SPEECH_EPOCHS), "--batch-size", "1", "--grad-accum", "16",
    "--keep-adapter", "--no-convert")
adapter = Path("models/lilly/listen-adapter")
assert (adapter / "trainer_state.json").is_file(), (
    f"no Trainer checkpoint at {adapter}: "
    f"{sorted(p.name for p in adapter.iterdir()) if adapter.is_dir() else 'missing'}")
OFF.check_trainproof()
run("zip", "-qr", "/kaggle/working/lilly-listen-half1.zip",
    "models/lilly/listen-adapter")
size = Path("/kaggle/working/lilly-listen-half1.zip").stat().st_size
assert size > 1_000_000, f"the zip is only {size} bytes"
print(f"lilly-listen-half1.zip — {size / 1048576:.0f} MB")
OFF.finish("half1", [
    "/kaggle/working/lilly-listen-half1.zip",
    "/kaggle/working/stdout.txt",
    "/kaggle/working/experiment_log.json",
])


**Done.** Download `lilly-listen-half1.zip` from the **Output** tab.

That zip is the Trainer checkpoint (adapter + optimizer + `trainer_state.json`),
not the app listener. Half 2 resumes it, then converts to `lilly-listen.zip`.

If this version was **cancelled**, the zip is gone even if Output still names it.
COMPLETE is what keeps the file. COMPLETE is still not install — half 2 converts
and scores WER first.
